# Ancestry & Batch Correction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/06_ancestry_batch_correction.ipynb)

**What this does:** Detects and removes confounding from population structure (ancestry) and technical batch effects, so discovered subtypes reflect biology rather than artifacts.

**Two independent corrections:**
- **Ancestry correction** — regress out ancestry PCs from pathway scores (Price et al., 2006)
- **Batch correction** — ComBat empirical Bayes adjustment (Johnson et al., 2007)

**Prerequisites:** [00_quick_demo.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/00_quick_demo.ipynb)

In [ ]:
# Install pathway-subtyping
!pip install -q pathway-subtyping==0.2.3

import pathway_subtyping
print(f"pathway-subtyping v{pathway_subtyping.__version__}")

## Part 1: Ancestry Correction

### 1.1 Simulate Ancestry-Confounded Data

We'll create synthetic pathway scores where some variation is driven by ancestry (population structure) rather than true biology.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_samples = 150
n_pathways = 8
n_variants = 200

# Simulate genotype matrix (samples × variants) with population structure
# 3 ancestry groups with different allele frequencies
ancestry_groups = np.repeat(["EUR", "EAS", "AFR"], 50)
genotype_matrix = np.zeros((n_samples, n_variants))

for i, group in enumerate(["EUR", "EAS", "AFR"]):
    mask = ancestry_groups == group
    # Group-specific allele frequencies
    freqs = np.random.beta(0.5 + i * 0.3, 0.5, n_variants)
    for j in range(n_variants):
        genotype_matrix[mask, j] = np.random.binomial(2, freqs[j], mask.sum())

genotype_df = pd.DataFrame(
    genotype_matrix,
    index=[f"SAMPLE_{i:03d}" for i in range(n_samples)],
    columns=[f"VAR_{j}" for j in range(n_variants)],
)

# Create pathway scores with ancestry confounding
pathway_names = [f"Pathway_{i}" for i in range(n_pathways)]
pathway_scores = pd.DataFrame(
    np.random.randn(n_samples, n_pathways),
    index=genotype_df.index,
    columns=pathway_names,
)

# Add ancestry-correlated signal to first 3 pathways
for i, group in enumerate(["EUR", "EAS", "AFR"]):
    mask = ancestry_groups == group
    pathway_scores.loc[mask, "Pathway_0"] += (i - 1) * 1.5
    pathway_scores.loc[mask, "Pathway_1"] += (i - 1) * 1.2
    pathway_scores.loc[mask, "Pathway_2"] += (i - 1) * 0.8

print(f"Genotype matrix: {genotype_df.shape}")
print(f"Pathway scores:  {pathway_scores.shape}")
print(f"Ancestry groups: {dict(zip(*np.unique(ancestry_groups, return_counts=True)))}")

### 1.2 Compute Ancestry PCs and Check Correlation

In [ ]:
from pathway_subtyping import compute_ancestry_pcs, compute_ancestry_correlation

# Compute ancestry PCs from genotype data
pcs = compute_ancestry_pcs(genotype_df, n_components=10, seed=42)

print(f"Ancestry PCs: {pcs.n_components} components")
print(f"Variance explained: {sum(pcs.explained_variance_ratio):.1%}")
print(f"  PC1: {pcs.explained_variance_ratio[0]:.1%}")
print(f"  PC2: {pcs.explained_variance_ratio[1]:.1%}")

# Check which pathways correlate with ancestry
corr = compute_ancestry_correlation(pathway_scores, pcs, n_pcs=3)
print(f"\nPathway-Ancestry correlations (Pearson r):")
print(corr.to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot ancestry PCs colored by group
colors = {"EUR": "tab:blue", "EAS": "tab:orange", "AFR": "tab:green"}
for group in ["EUR", "EAS", "AFR"]:
    mask = ancestry_groups == group
    axes[0].scatter(
        pcs.components.iloc[mask, 0], pcs.components.iloc[mask, 1],
        label=group, c=colors[group], s=30, alpha=0.7,
    )
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].set_title("Ancestry PCs (genotype-derived)")
axes[0].legend()

# Plot pathway scores — confounded pathways separate by ancestry
for group in ["EUR", "EAS", "AFR"]:
    mask = ancestry_groups == group
    axes[1].scatter(
        pathway_scores.loc[mask, "Pathway_0"], pathway_scores.loc[mask, "Pathway_1"],
        label=group, c=colors[group], s=30, alpha=0.7,
    )
axes[1].set_xlabel("Pathway_0 score")
axes[1].set_ylabel("Pathway_1 score")
axes[1].set_title("Pathway scores (ancestry-confounded)")
axes[1].legend()

plt.tight_layout()
plt.show()

### 1.3 Correct Pathway Scores

In [ ]:
from pathway_subtyping import adjust_pathway_scores, AncestryMethod

# Regress out ancestry PCs
result = adjust_pathway_scores(
    pathway_scores, pcs,
    method=AncestryMethod.REGRESS_OUT,
    n_pcs=5,
)

print(result.format_report())

# Compare correlations before vs after
corr_after = compute_ancestry_correlation(result.adjusted_scores, pcs, n_pcs=3)
print("\nCorrelation with ancestry AFTER correction:")
print(corr_after.to_string())

### 1.4 Cluster and Test Independence

In [ ]:
from pathway_subtyping import run_clustering, check_ancestry_independence

# Cluster corrected scores
clustering = run_clustering(result.adjusted_scores.values, n_clusters=3, seed=42)

# Test if clusters are independent of ancestry
report = check_ancestry_independence(clustering.labels, pcs)
print(report.format_report())
print(f"\nClusters independent of ancestry: {report.overall_independence_passed}")

---

## Part 2: Batch Correction

### 2.1 Simulate Batch Effects

In [ ]:
from pathway_subtyping import detect_batch_effects

# Add batch effects to pathway scores
batch_labels = np.array(["Site_A"] * 50 + ["Site_B"] * 50 + ["Site_C"] * 50)
batched_scores = pathway_scores.copy()

# Site_A: baseline
# Site_B: mean shift on all pathways
batched_scores.iloc[50:100] += 0.8
# Site_C: mean shift + variance change
batched_scores.iloc[100:150] += np.random.randn(50, n_pathways) * 0.5 - 0.5

# Detect batch effects
report = detect_batch_effects(batched_scores, batch_labels, batch_variable="sequencing_site")
print(report.format_report())

### 2.2 Correct Batch Effects

In [ ]:
from pathway_subtyping import correct_batch_effects, BatchCorrectionMethod

# ComBat correction (recommended)
combat_result = correct_batch_effects(
    batched_scores, batch_labels,
    method=BatchCorrectionMethod.COMBAT,
    batch_variable="sequencing_site",
)

print(combat_result.format_report())

### 2.3 Validate Correction

In [ ]:
from pathway_subtyping import validate_batch_correction

val = validate_batch_correction(
    batched_scores, combat_result.corrected_scores,
    batch_labels,
)

print("Validation Results:")
for k, v in val.items():
    print(f"  {k}: {v}")

In [ ]:
# Compare methods
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
from sklearn.decomposition import PCA

datasets = {
    "Before Correction": batched_scores,
    "ComBat": combat_result.corrected_scores,
    "Mean-Center": correct_batch_effects(
        batched_scores, batch_labels, method=BatchCorrectionMethod.MEAN_CENTER
    ).corrected_scores,
}

batch_colors = {"Site_A": "tab:blue", "Site_B": "tab:orange", "Site_C": "tab:green"}

for ax, (title, scores) in zip(axes, datasets.items()):
    pca = PCA(n_components=2, random_state=42)
    X = pca.fit_transform(scores.values)

    for batch in ["Site_A", "Site_B", "Site_C"]:
        mask = batch_labels == batch
        ax.scatter(X[mask, 0], X[mask, 1], label=batch, c=batch_colors[batch], s=30, alpha=0.7)

    ax.set_title(title)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
    ax.legend(fontsize=8)

plt.suptitle("Batch Correction Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

| Step | Function | When to Use |
|------|----------|-------------|
| Detect ancestry confounding | `compute_ancestry_pcs()` + `compute_ancestry_correlation()` | Multi-ethnic cohorts |
| Correct ancestry | `adjust_pathway_scores()` | Pathways correlated with ancestry PCs |
| Verify independence | `check_ancestry_independence()` | After ancestry correction, before publishing |
| Detect batch effects | `detect_batch_effects()` | Multi-site or multi-batch studies |
| Correct batch | `correct_batch_effects()` | Significant batch effects detected |
| Validate correction | `validate_batch_correction()` | After any correction |

## Next Steps

- **Sensitivity analysis:** [07_sensitivity_analysis.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/07_sensitivity_analysis.ipynb) — test robustness
- **API reference:** [Ancestry API](https://github.com/topmist-admin/pathway-subtyping-framework/blob/main/docs/api/ancestry.md) | [Batch Correction API](https://github.com/topmist-admin/pathway-subtyping-framework/blob/main/docs/api/batch_correction.md)

---
*Built with [pathway-subtyping](https://pypi.org/project/pathway-subtyping/). Disease-agnostic. Open source.*